In [2]:
import requests
import os
import time

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/146.0.0.0 Safari/537.36",
    "Referer": "https://www.musinsa.com/",
    "Origin": "https://www.musinsa.com",
}

# 테스트용 상품번호 (filluminate 바지)
TEST_GOODS_NO = "3791988"

# 이미지 저장 폴더
SAVE_DIR = f"review_images/{TEST_GOODS_NO}"
os.makedirs(SAVE_DIR, exist_ok=True)

# ==========================================
# Step 1: 리뷰 1페이지 가져오기
# ==========================================

url = "https://goods.musinsa.com/api2/review/v1/view/list"
params = {
    "page":              0,
    "pageSize":          10,
    "goodsNo":           TEST_GOODS_NO,
    "sort":              "up_cnt_desc",
    "selectedSimilarNo": TEST_GOODS_NO,
    "myFilter":          "false",
    "hasPhoto":          "true",   # 사진 있는 리뷰만
    "isExperience":      "false",
}

r = requests.get(url, headers=headers, params=params, timeout=10)
review_list = r.json().get("data", {}).get("list", [])
print(f"리뷰 {len(review_list)}개 가져옴\n")

# ==========================================
# Step 2: 이미지 다운로드
# ==========================================

total_images = 0
start_time = time.time()

for review in review_list:
    images = review.get("images") or []
    review_no = review.get("no", "unknown")

    for i, img in enumerate(images):
        img_url = img.get("imageUrl") or img.get("url") or img.get("thumbnailImageUrl")

        if not img_url:
            print(f"  [리뷰 {review_no}] 이미지 URL 없음 - img 구조: {img.keys()}")
            continue

        # https:// 없으면 붙여주기
        if img_url.startswith("//"):
            img_url = "https:" + img_url
        elif img_url.startswith("/"):
            img_url = "https://image.msscdn.net" + img_url

        try:
            img_response = requests.get(img_url, headers=headers, timeout=10)

            if img_response.status_code == 200:
                filename = f"{SAVE_DIR}/review_{review_no}_{i}.jpg"
                with open(filename, "wb") as f:
                    f.write(img_response.content)
                total_images += 1
                print(f"  [리뷰 {review_no}] 이미지 {i+1} 다운로드 완료 ({len(img_response.content)/1024:.1f} KB)")
            else:
                print(f"  [리뷰 {review_no}] 이미지 {i+1} 실패: {img_response.status_code}")

        except Exception as e:
            print(f"  [리뷰 {review_no}] 오류: {e}")

        time.sleep(0.2)

elapsed = time.time() - start_time
print(f"\n총 {total_images}개 이미지 다운로드 완료")
print(f"소요 시간: {elapsed:.1f}초")
print(f"저장 위치: {SAVE_DIR}/")

리뷰 10개 가져옴

  [리뷰 81767333] 이미지 1 다운로드 완료 (137.4 KB)
  [리뷰 81767333] 이미지 2 다운로드 완료 (101.5 KB)
  [리뷰 83446456] 이미지 1 다운로드 완료 (239.8 KB)
  [리뷰 83114659] 이미지 1 다운로드 완료 (155.1 KB)
  [리뷰 79476621] 이미지 1 다운로드 완료 (127.4 KB)
  [리뷰 79476621] 이미지 2 다운로드 완료 (191.1 KB)
  [리뷰 79476621] 이미지 3 다운로드 완료 (201.7 KB)
  [리뷰 79476621] 이미지 4 다운로드 완료 (200.9 KB)
  [리뷰 79476621] 이미지 5 다운로드 완료 (135.0 KB)
  [리뷰 81410380] 이미지 1 다운로드 완료 (117.4 KB)
  [리뷰 80873287] 이미지 1 다운로드 완료 (449.3 KB)
  [리뷰 80873287] 이미지 2 다운로드 완료 (169.8 KB)
  [리뷰 80873287] 이미지 3 다운로드 완료 (181.9 KB)
  [리뷰 83676851] 이미지 1 다운로드 완료 (225.7 KB)
  [리뷰 83676851] 이미지 2 다운로드 완료 (304.7 KB)
  [리뷰 83676851] 이미지 3 다운로드 완료 (317.7 KB)
  [리뷰 83676828] 이미지 1 다운로드 완료 (317.6 KB)
  [리뷰 83676828] 이미지 2 다운로드 완료 (225.6 KB)
  [리뷰 83676828] 이미지 3 다운로드 완료 (304.5 KB)
  [리뷰 83548653] 이미지 1 다운로드 완료 (200.0 KB)
  [리뷰 83655223] 이미지 1 다운로드 완료 (124.0 KB)

총 21개 이미지 다운로드 완료
소요 시간: 7.8초
저장 위치: review_images/3791988/
